# 01 Preprocessing and Splits

Self-contained preprocessing notebook. It defines the WESAD preprocessing functions here and writes the shared model inputs.


## 1. Imports and configuration


In [ ]:
from pathlib import Path

cwd = Path.cwd()
if cwd.name == "notebooks":
    PROJECT_ROOT = cwd.parent
elif (cwd / "data").exists() and (cwd / "notebooks").exists():
    PROJECT_ROOT = cwd
elif (cwd / "wesad_stress_project").exists():
    PROJECT_ROOT = cwd / "wesad_stress_project"
else:
    PROJECT_ROOT = cwd
PROJECT_ROOT


## 2. Preprocessing functions


In [ ]:
from __future__ import annotations

import json
import pickle
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Tuple

import torch
import numpy as np
import pandas as pd
from scipy.signal import resample_poly
from sklearn.preprocessing import StandardScaler


TRAIN_SUBJECTS = [
    "S3",
    "S4",
    "S6",
    "S7",
    "S8",
    "S9",
    "S10",
    "S13",
    "S16",
    "S17",
]
VALIDATION_SUBJECTS = ["S5", "S15"]
TEST_SUBJECTS = ["S2", "S11", "S14"]

SPLIT_SUBJECTS = {
    "train": TRAIN_SUBJECTS,
    "validation": VALIDATION_SUBJECTS,
    "test": TEST_SUBJECTS,
}

SEQUENCE_CHANNELS = ["BVP", "EDA", "TEMP", "ACC_x", "ACC_y", "ACC_z"]
ACCEPTED_LABELS = {1, 2, 3}
BINARY_LABEL_MAP = {1: 0, 2: 1, 3: 0}

TARGET_HZ = 32
WINDOW_SECONDS = 30
STRIDE_SECONDS = 15
WINDOW_SAMPLES = TARGET_HZ * WINDOW_SECONDS
STRIDE_SAMPLES = TARGET_HZ * STRIDE_SECONDS

WRIST_SAMPLE_RATES = {
    "BVP": 64,
    "EDA": 4,
    "TEMP": 4,
    "ACC": 32,
}
LABEL_HZ = 700


@dataclass(frozen=True)
class ProcessedSplit:
    X_sequence: np.ndarray
    y: np.ndarray
    metadata: pd.DataFrame


def load_subject_pickle(data_root: Path, subject_id: str) -> dict:
    path = data_root / subject_id / f"{subject_id}.pkl"
    with path.open("rb") as handle:
        return pickle.load(handle, encoding="latin1")


def _interp_to_target(values: np.ndarray, source_hz: int, target_times: np.ndarray) -> np.ndarray:
    values = np.asarray(values)
    if values.ndim == 1:
        values = values[:, None]

    source_times = np.arange(values.shape[0], dtype=np.float64) / float(source_hz)
    channels = [
        np.interp(target_times, source_times, values[:, channel]).astype(np.float32)
        for channel in range(values.shape[1])
    ]
    return np.column_stack(channels)


def _downsample_bvp_to_32hz(values: np.ndarray, n_samples: int) -> np.ndarray:
    values = np.asarray(values, dtype=np.float32).reshape(-1)
    downsampled = resample_poly(values, up=1, down=2).astype(np.float32)
    if downsampled.shape[0] < n_samples:
        source_times = np.arange(downsampled.shape[0], dtype=np.float64) / float(TARGET_HZ)
        target_times = np.arange(n_samples, dtype=np.float64) / float(TARGET_HZ)
        downsampled = np.interp(target_times, source_times, downsampled).astype(np.float32)
    return downsampled[:n_samples, None]


def align_subject_to_32hz(subject_data: dict) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    wrist = subject_data["signal"]["wrist"]
    label = np.asarray(subject_data["label"]).reshape(-1)

    durations = [
        wrist["BVP"].shape[0] / WRIST_SAMPLE_RATES["BVP"],
        wrist["EDA"].shape[0] / WRIST_SAMPLE_RATES["EDA"],
        wrist["TEMP"].shape[0] / WRIST_SAMPLE_RATES["TEMP"],
        wrist["ACC"].shape[0] / WRIST_SAMPLE_RATES["ACC"],
        label.shape[0] / LABEL_HZ,
    ]
    duration = min(durations)
    n_samples = int(np.floor(duration * TARGET_HZ))
    target_times = np.arange(n_samples, dtype=np.float64) / float(TARGET_HZ)

    bvp = _downsample_bvp_to_32hz(wrist["BVP"], n_samples)
    eda = _interp_to_target(wrist["EDA"], WRIST_SAMPLE_RATES["EDA"], target_times)
    temp = _interp_to_target(wrist["TEMP"], WRIST_SAMPLE_RATES["TEMP"], target_times)
    acc = _interp_to_target(wrist["ACC"], WRIST_SAMPLE_RATES["ACC"], target_times)

    label_indices = np.minimum((target_times * LABEL_HZ).astype(np.int64), label.shape[0] - 1)
    aligned_labels = label[label_indices].astype(np.int16)
    aligned = np.column_stack([bvp[:, 0], eda[:, 0], temp[:, 0], acc]).astype(np.float32)
    return aligned, aligned_labels, target_times.astype(np.float32)


def continuous_label_segments(labels: np.ndarray, accepted_labels: Iterable[int] = ACCEPTED_LABELS) -> List[Tuple[int, int, int]]:
    accepted = set(accepted_labels)
    segments: List[Tuple[int, int, int]] = []
    start = None
    active_label = None

    for index, value in enumerate(labels):
        label = int(value)
        if label not in accepted:
            if start is not None:
                segments.append((start, index, active_label))
                start = None
                active_label = None
            continue

        if start is None:
            start = index
            active_label = label
        elif label != active_label:
            segments.append((start, index, active_label))
            start = index
            active_label = label

    if start is not None:
        segments.append((start, len(labels), active_label))

    return segments


def create_subject_windows(subject_id: str, split: str, subject_data: dict, start_window_id: int) -> Tuple[np.ndarray, np.ndarray, pd.DataFrame]:
    aligned, labels, times = align_subject_to_32hz(subject_data)
    windows = []
    y = []
    rows = []
    window_id = start_window_id

    for segment_start, segment_end, original_label in continuous_label_segments(labels):
        if segment_end - segment_start < WINDOW_SAMPLES:
            continue
        for start in range(segment_start, segment_end - WINDOW_SAMPLES + 1, STRIDE_SAMPLES):
            end = start + WINDOW_SAMPLES
            windows.append(aligned[start:end])
            y.append(BINARY_LABEL_MAP[original_label])
            rows.append(
                {
                    "window_id": window_id,
                    "subject_id": subject_id,
                    "split": split,
                    "start_time": float(times[start]),
                    "original_label": int(original_label),
                    "binary_label": int(BINARY_LABEL_MAP[original_label]),
                }
            )
            window_id += 1

    if not windows:
        return (
            np.empty((0, WINDOW_SAMPLES, len(SEQUENCE_CHANNELS)), dtype=np.float32),
            np.empty((0,), dtype=np.float32),
            pd.DataFrame(rows),
        )

    return np.stack(windows).astype(np.float32), np.asarray(y, dtype=np.float32), pd.DataFrame(rows)


def _fit_transform_sequences(
    X_train: np.ndarray, splits: Dict[str, np.ndarray]
) -> Tuple[Dict[str, np.ndarray], StandardScaler]:
    scaler = StandardScaler()
    scaler.fit(X_train.reshape(-1, X_train.shape[-1]))
    transformed = {}
    for split, X in splits.items():
        transformed[split] = scaler.transform(X.reshape(-1, X.shape[-1])).reshape(X.shape).astype(np.float32)
    return transformed, scaler


def run_preprocessing(project_root: Path) -> Dict[str, object]:
    data_root = project_root / "data" / "WESAD" / "WESAD"
    sequence_dir = project_root / "data" / "processed" / "sequence"
    metadata_dir = project_root / "data" / "processed" / "metadata"
    artifact_dir = project_root / "artifacts" / "preprocessing"
    for directory in [sequence_dir, metadata_dir, artifact_dir]:
        directory.mkdir(parents=True, exist_ok=True)

    raw_sequences: Dict[str, List[np.ndarray]] = {split: [] for split in SPLIT_SUBJECTS}
    labels: Dict[str, List[np.ndarray]] = {split: [] for split in SPLIT_SUBJECTS}
    metadata_frames: Dict[str, List[pd.DataFrame]] = {split: [] for split in SPLIT_SUBJECTS}
    next_window_id = 0

    for split, subjects in SPLIT_SUBJECTS.items():
        for subject_id in subjects:
            subject_data = load_subject_pickle(data_root, subject_id)
            X_subject, y_subject, meta_subject = create_subject_windows(
                subject_id, split, subject_data, next_window_id
            )
            if len(meta_subject):
                next_window_id = int(meta_subject["window_id"].max()) + 1
            raw_sequences[split].append(X_subject)
            labels[split].append(y_subject)
            metadata_frames[split].append(meta_subject)

    X_raw = {
        split: np.concatenate(parts, axis=0).astype(np.float32)
        for split, parts in raw_sequences.items()
    }
    y = {
        split: np.concatenate(parts, axis=0).astype(np.float32)
        for split, parts in labels.items()
    }
    metadata = {
        split: pd.concat(parts, ignore_index=True)
        for split, parts in metadata_frames.items()
    }

    assert set(TRAIN_SUBJECTS).isdisjoint(VALIDATION_SUBJECTS)
    assert set(TRAIN_SUBJECTS).isdisjoint(TEST_SUBJECTS)
    assert set(VALIDATION_SUBJECTS).isdisjoint(TEST_SUBJECTS)

    X_sequence, sequence_scaler = _fit_transform_sequences(X_raw["train"], X_raw)

    for split in SPLIT_SUBJECTS:
        assert X_raw[split].shape[1:] == (WINDOW_SAMPLES, len(SEQUENCE_CHANNELS))
        assert X_sequence[split].shape[1:] == (WINDOW_SAMPLES, len(SEQUENCE_CHANNELS))
        assert np.isfinite(X_raw[split]).all()
        assert np.isfinite(X_sequence[split]).all()
        assert set(np.unique(y[split]).astype(float).tolist()) == {0.0, 1.0}
        assert len(metadata[split]) == len(y[split])
        assert np.array_equal(metadata[split]["binary_label"].to_numpy(), y[split].astype(int))

        torch.save(torch.from_numpy(X_raw[split]), sequence_dir / f"X_{split}_raw.pt")
        torch.save(torch.from_numpy(X_sequence[split]), sequence_dir / f"X_{split}.pt")
        torch.save(torch.from_numpy(y[split]), sequence_dir / f"y_{split}.pt")
        metadata[split].to_csv(metadata_dir / f"windows_{split}.csv", index=False)

    all_metadata = pd.concat(metadata.values(), ignore_index=True)
    all_metadata.to_csv(metadata_dir / "windows_all.csv", index=False)
    all_metadata.to_csv(metadata_dir / "window_metadata.csv", index=False)

    with (artifact_dir / "sequence_channels.json").open("w", encoding="utf-8") as handle:
        json.dump(SEQUENCE_CHANNELS, handle, indent=2)
    with (artifact_dir / "split_subjects.json").open("w", encoding="utf-8") as handle:
        json.dump(SPLIT_SUBJECTS, handle, indent=2)
    with (artifact_dir / "splits.json").open("w", encoding="utf-8") as handle:
        json.dump(SPLIT_SUBJECTS, handle, indent=2)
    with (artifact_dir / "preprocessing_config.json").open("w", encoding="utf-8") as handle:
        json.dump(
            {
                "target_hz": TARGET_HZ,
                "window_seconds": WINDOW_SECONDS,
                "window_samples": WINDOW_SAMPLES,
                "stride_seconds": STRIDE_SECONDS,
                "stride_samples": STRIDE_SAMPLES,
                "sequence_channels": SEQUENCE_CHANNELS,
                "accepted_labels": sorted(ACCEPTED_LABELS),
                "binary_label_map": BINARY_LABEL_MAP,
            },
            handle,
            indent=2,
        )

    np.savez(
        artifact_dir / "sequence_scaler.npz",
        mean=sequence_scaler.mean_,
        scale=sequence_scaler.scale_,
        var=sequence_scaler.var_,
    )
    with (artifact_dir / "sequence_scaler.pkl").open("wb") as handle:
        pickle.dump(sequence_scaler, handle)

    return {
        "sequence_shapes": {split: tuple(X_sequence[split].shape) for split in SPLIT_SUBJECTS},
        "label_counts": {
            split: pd.Series(y[split]).value_counts().sort_index().to_dict()
            for split in SPLIT_SUBJECTS
        },
        "metadata_rows": {split: int(len(metadata[split])) for split in SPLIT_SUBJECTS},
    }


## 3. Fixed subject split and window configuration


In [ ]:
SPLIT_SUBJECTS, TARGET_HZ, WINDOW_SECONDS, WINDOW_SAMPLES, STRIDE_SECONDS, STRIDE_SAMPLES, SEQUENCE_CHANNELS


## 4. Load each subject separately


`run_preprocessing` loads each subject pickle independently from the fixed split.


## 5. Extract BVP, EDA, TEMP and ACC


The notebook uses wrist BVP, EDA, TEMP, and 3-axis ACC only.


## 6. Align all signals to 32 Hz


BVP is downsampled from 64 Hz to 32 Hz with polyphase resampling. EDA and TEMP are linearly interpolated from 4 Hz, ACC remains at 32 Hz, and labels are sampled from the 700 Hz label stream.


## 7. Keep labels 1, 2 and 3


Only baseline, stress, and amusement are retained.


## 8. Create continuous condition segments


Windows are generated only within contiguous accepted-label segments.


## 9. Create 30-second windows with 15-second stride


Each sequence window has shape `(960, 6)`.


## 10. Map labels to stress/non-stress


Label 2 maps to `1`; labels 1 and 3 map to `0`.


## 11. Fit sequence scaler on training subjects only and save outputs


In [ ]:
summary = run_preprocessing(PROJECT_ROOT)
summary


## 12. Verify saved artifacts


In [ ]:
for path in [
    PROJECT_ROOT / "data" / "processed" / "sequence" / "X_train_raw.pt",
    PROJECT_ROOT / "data" / "processed" / "sequence" / "X_train.pt",
    PROJECT_ROOT / "data" / "processed" / "sequence" / "y_train.pt",
    PROJECT_ROOT / "data" / "processed" / "metadata" / "windows_all.csv",
    PROJECT_ROOT / "artifacts" / "preprocessing" / "sequence_channels.json",
    PROJECT_ROOT / "artifacts" / "preprocessing" / "sequence_scaler.pkl",
]:
    print(path.relative_to(PROJECT_ROOT), path.exists())
